<a href="https://colab.research.google.com/github/rantawadeesritakorn-tech/Project-Hotel/blob/%E0%B8%84%E0%B8%99%E0%B8%97%E0%B8%B5%E0%B9%88-5/%E0%B8%84%E0%B8%99%E0%B8%97%E0%B8%B5%E0%B9%885.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ส่วนที่ 9 — นำเสนอผลด้วยกราฟและ Data Storytelling


In [ ]:
# ===== ตั้งค่าฟอนต์ภาษาไทยให้ matplotlib
import os
import shutil

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# 1. ติดตั้งฟอนต์ไทย
!apt-get -qq -y install fonts-thai-tlwg > /dev/null

# 2. ล้าง cache ของ matplotlib เพื่อให้มองเห็นฟอนต์ที่เพิ่งติดตั้ง
cache_dir = matplotlib.get_cachedir()
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

# 3. ลงทะเบียนฟอนต์ Loma เข้ากับ FontManager
for font_path in ["/usr/share/fonts/truetype/tlwg/Loma.ttf",
                  "/usr/share/fonts/opentype/tlwg/Loma.otf"]:
    if os.path.exists(font_path):
        fm.fontManager.addfont(font_path)
        break

# 4. ตั้งเป็นฟอนต์หลัก
plt.rcParams["font.family"] = "Loma"
plt.rcParams["axes.unicode_minus"] = False      # กันเครื่องหมายลบกลายเป็นสี่เหลี่ยม
sns.set_theme(style="whitegrid", font="Loma")   # ธีมของ seaborn + ฟอนต์ไทย

print("ฟอนต์ที่ใช้:", plt.rcParams["font.family"])

ฟอนต์ที่ใช้: ['Loma']


### ขั้นที่ 1 — Context


In [ ]:
print(f"กำลังวิเคราะห์ใบจอง {len(active):,} ใบ จากทั้งหมด {len(bookings):,} ใบ")
print(f"ครอบคลุม {active['nationality'].nunique()} สัญชาติ "
      f"{active['room_type'].nunique()} ประเภทห้อง "
      f"{active['channel'].nunique()} ช่องทางการจอง")
print()
print("คำถามที่ต้องการตอบ")
print("  1. ห้องประเภทไหนควรลงทุนเพิ่ม")
print("  2. ช่องทางไหนทำเงินให้โรงแรมได้จริงหลังหักค่าคอมมิชชัน")
print("  3. ฤดูกาลมีผลต่อราคาและปริมาณการจองแค่ไหน")

NameError: name 'active' is not defined

### ขั้นที่ 2 — Evidence

#### กราฟที่ 1 : ห้องประเภทไหนทำเงินได้มากที่สุด (Bar Chart)


In [ ]:
top_type = by_room_type.iloc[0]

# สีเน้นเฉพาะห้องที่ทำรายได้สูงสุด แท่งอื่นใช้สีเทา
colors = ["firebrick" if t == top_type["room_type"] else "lightgray"
          for t in by_room_type["room_type"]]

plt.figure(figsize=(10, 5))
bars = plt.bar(by_room_type["room_type"], by_room_type["net_revenue"], color=colors)

plt.title(f"ห้อง {top_type['room_type']} ทำรายได้สุทธิสูงสุด "
          f"{top_type['net_revenue']/1e6:.2f} ล้านบาท ใน 6 เดือน",
          fontsize=13, fontweight="bold")
plt.xlabel("ประเภทห้องพัก")
plt.ylabel("รายได้สุทธิ (บาท)")

for bar, v in zip(bars, by_room_type["net_revenue"]):
    plt.text(bar.get_x() + bar.get_width()/2, v, f"{v:,.0f}",
             ha="center", va="bottom", fontsize=9)

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

#### กราฟที่ 2 : ยอดรวมกับยอดต่อคืนให้คำตอบคนละแบบ (Subplot)


In [ ]:
by_night = by_room_type.sort_values("net_per_night", ascending=False)
win_total = by_room_type.iloc[0]["room_type"]
win_night = by_night.iloc[0]["room_type"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ซ้าย: ยอดรวม
c1 = ["firebrick" if t == win_total else "lightgray" for t in by_room_type["room_type"]]
axes[0].bar(by_room_type["room_type"], by_room_type["net_revenue"], color=c1)
axes[0].set_title(f"ถ้าดูยอดรวม : {win_total} ชนะ", fontsize=12, fontweight="bold")
axes[0].set_xlabel("ประเภทห้องพัก")
axes[0].set_ylabel("รายได้สุทธิรวม (บาท)")

# ขวา: ยอดต่อคืน
c2 = ["seagreen" if t == win_night else "lightgray" for t in by_night["room_type"]]
axes[1].bar(by_night["room_type"], by_night["net_per_night"], color=c2)
axes[1].set_title(f"แต่ถ้าดูต่อคืน : {win_night} ชนะ", fontsize=12, fontweight="bold")
axes[1].set_xlabel("ประเภทห้องพัก")
axes[1].set_ylabel("รายได้สุทธิต่อคืน (บาท)")

plt.suptitle("ตัวเลขรวมกับตัวเลขต่อหน่วยให้คำตอบคนละแบบ", y=1.03,
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"{win_total} ทำเงินรวมมากที่สุดเพราะมีห้องเยอะและจองบ่อย")
print(f"แต่ {win_night} ทำเงินต่อคืนสูงกว่า "
      f"{by_night.iloc[0]['net_per_night'] / by_night.iloc[-1]['net_per_night']:.1f} เท่า"
      " ของห้องที่ต่ำสุด")

#### กราฟที่ 3 : ฤดูกาลดันราคาขึ้นได้จริงไหม (`pandas.plot()` + `secondary_y`)

In [ ]:
month_plot = by_month.set_index("booking_month")[["occupancy_%", "avg_rate"]]
month_plot.columns = ["อัตราการเข้าพัก (%)", "ราคาเฉลี่ยต่อคืน (บาท)"]

ax = month_plot.plot(figsize=(11, 5), marker="o", secondary_y="ราคาเฉลี่ยต่อคืน (บาท)",
                     color=["steelblue", "firebrick"])
gap = by_month["avg_rate"].max() / by_month["avg_rate"].min() - 1
ax.set_title(f"ราคาห้องช่วง High season สูงกว่าช่วงต่ำสุด {gap*100:.0f}% "
             "โดยที่คนไม่ได้ลดลง", fontsize=13, fontweight="bold")
ax.set_xlabel("เดือน")
ax.set_ylabel("อัตราการเข้าพัก (%)")
ax.right_ax.set_ylabel("ราคาเฉลี่ยต่อคืน (บาท)")
plt.tight_layout()
plt.show()

#### กราฟที่ 4 : ลูกค้าพักกี่คืน (Histogram + ทดลองปรับ `bins`)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, n_bins in zip(axes, [4, 12, 40]):
    ax.hist(active["nights"], bins=n_bins, color="mediumpurple", edgecolor="white")
    ax.set_title(f"bins = {n_bins}")
    ax.set_xlabel("จำนวนคืนที่เข้าพัก")
    ax.set_ylabel("จำนวนใบจอง")

short = (active["nights"] <= 2).mean() * 100
plt.suptitle(f"การจอง {short:.0f}% เป็นการพักสั้นเพียง 1-2 คืน", y=1.04,
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("bins น้อยเกินไป (4) เห็นภาพรวมหยาบจนไม่รู้ว่าคนพัก 1 คืนหรือ 2 คืนมากกว่ากัน")
print("bins มากเกินไป (40) เห็นแต่ช่องว่าง เพราะจำนวนคืนเป็นเลขจำนวนเต็ม")
print("bins = 12 กำลังพอดีกับข้อมูลชุดนี้")

#### กราฟที่ 5 : ราคาต่อคืนกระจายตัวอย่างไรในแต่ละประเภทห้อง (`sns.boxplot()` + `hue`)


In [ ]:
plt.figure(figsize=(11, 6))
sns.boxplot(data=active, x="room_type", y="rate_per_night", hue="season",
            palette={"High": "firebrick", "Normal": "lightgray", "Low": "steelblue"})

plt.title("ทุกประเภทห้องขายได้แพงขึ้นในช่วง High season "
          "และห้อง Suite มีช่วงราคากว้างที่สุด", fontsize=13, fontweight="bold")
plt.xlabel("ประเภทห้องพัก")
plt.ylabel("ราคาต่อคืน (บาท)")
plt.legend(title="ฤดูกาล")
plt.tight_layout()
plt.show()

print(active.groupby(["room_type", "season"])["rate_per_night"]
      .median().unstack().round(0))

#### กราฟที่ 6 : ตัวแปรไหนสัมพันธ์กับยอดเงินบ้าง (`sns.heatmap()` + `.corr()`)

Correlation มีค่าระหว่าง -1 ถึง 1 — ใกล้ +1 คือไปทางเดียวกัน ใกล้ -1 คือตรงข้ามกัน

In [ ]:
numeric_cols = ["nights", "adults", "lead_time_days", "rate_per_night",
                "room_charge", "commission", "total_price", "net_revenue"]
corr = active[numeric_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title("จำนวนคืนคือตัวแปรที่สัมพันธ์กับยอดเงินมากที่สุด ไม่ใช่ราคาต่อคืน",
          fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("ค่า correlation กับ total_price เรียงจากมากไปน้อย")
print(corr["total_price"].drop("total_price").sort_values(ascending=False).round(2))
print()
print("ข้อควรระวัง: correlation บอกแค่ความสัมพันธ์เชิงเส้น ไม่ได้แปลว่าเป็นเหตุเป็นผลกัน")

#### กราฟที่ 7 : ช่องทางไหนทำเงินได้จริง (Grouped Bar)

ยอดขายที่เห็นไม่ใช่เงินที่โรงแรมได้จริง เพราะ OTA หักค่าคอมมิชชัน 15%

In [ ]:
import numpy as np

ch = by_channel.sort_values("gross_revenue", ascending=False)
x = np.arange(len(ch))
w = 0.38
ota = ch[ch["channel"] == "OTA"].iloc[0]

plt.figure(figsize=(10, 5))
plt.bar(x - w/2, ch["gross_revenue"], w, label="ยอดขายรวม", color="lightsteelblue")
plt.bar(x + w/2, ch["net_revenue"], w, label="รายได้สุทธิหลังหักค่าคอม",
        color="indianred")

plt.xticks(x, ch["channel"])
plt.title(f"OTA หักค่าคอมมิชชันไป {ota['commission_paid']:,.0f} บาท "
          f"คิดเป็น {ota['commission_pct']:.0f}% ของยอดขาย",
          fontsize=13, fontweight="bold")
plt.xlabel("ช่องทางการจอง")
plt.ylabel("รายได้ (บาท)")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

#### แดชบอร์ดสรุป : รวม 4 มุมมองไว้ในภาพเดียว (`plt.subplots(2, 2)`)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# (0,0) รายได้สุทธิแยกประเภทห้อง
axes[0, 0].bar(by_room_type["room_type"], by_room_type["net_revenue"],
               color="steelblue")
axes[0, 0].set_title("รายได้สุทธิแยกตามประเภทห้อง")
axes[0, 0].set_ylabel("บาท")

# (0,1) อัตราการเข้าพักรายเดือน
axes[0, 1].plot(by_month["booking_month"], by_month["occupancy_%"],
                marker="o", color="firebrick")
axes[0, 1].axhline(75.1, color="gray", linestyle="--", linewidth=1)
axes[0, 1].set_title("อัตราการเข้าพักรายเดือน (เส้นประ = ตลาดกรุงเทพ 75.1%)")
axes[0, 1].set_ylabel("%")
axes[0, 1].tick_params(axis="x", rotation=30)

# (1,0) สัดส่วนช่องทางการจอง
axes[1, 0].barh(ch["channel"], ch["bookings"], color="seagreen")
axes[1, 0].set_title("จำนวนใบจองแยกตามช่องทาง")
axes[1, 0].set_xlabel("จำนวนใบจอง")

# (1,1) 6 สัญชาติที่ทำรายได้สูงสุด
top_nat = by_nat.head(6).sort_values("revenue")
axes[1, 1].barh(top_nat["nationality"], top_nat["revenue"], color="darkorange")
axes[1, 1].set_title("6 สัญชาติที่ทำรายได้สูงสุด")
axes[1, 1].set_xlabel("รายได้ (บาท)")

plt.suptitle("สรุปภาพรวมผลประกอบการโรงแรม มกราคม - มิถุนายน 2025",
             y=1.01, fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

### ขั้นที่ 3 — Conclusion : สรุปเป็นข้อเสนอที่ทำได้จริง

การวิเคราะห์จะมีค่าก็ต่อเมื่อจบด้วย "แล้วเจ้าของโรงแรมควรทำอะไรต่อ"
ไม่ใช่จบแค่ "ตัวเลขเป็นแบบนี้"

In [ ]:
print("=" * 62)
print("ข้อเสนอต่อผู้บริหารโรงแรม จากข้อมูล 6 เดือน")
print("=" * 62)
print(f"1. ห้อง {win_night} ให้รายได้ต่อคืนสูงสุด "
      f"({by_night.iloc[0]['net_per_night']:,.0f} บาท/คืน) "
      f"แต่มีเพียง {int(by_night.iloc[0]['total_nights'] / by_night.iloc[0]['avg_nights'])} "
      "ครั้งที่ขายได้")
print(f"   -> ควรพิจารณาเพิ่มห้องประเภทนี้ หรือปรับห้องที่ขายไม่ดีมาเป็นประเภทนี้")
print()
print(f"2. OTA จ่ายค่าคอมไป {ota['commission_paid']:,.0f} บาท ใน 6 เดือน")
print(f"   -> ถ้าดันลูกค้าเก่าเพียง 20% ให้จองผ่านเว็บไซต์เอง จะประหยัดได้ราว "
      f"{ota['commission_paid'] * 0.2:,.0f} บาท")
print()
print(f"3. ราคาช่วง High season สูงกว่าช่วงต่ำสุด {gap*100:.0f}% "
      "โดยที่จำนวนใบจองไม่ได้ลดลงตาม")
print("   -> ยังมีพื้นที่ปรับราคาขึ้นได้อีกในช่วงฤดูท่องเที่ยว")
print()
# จำนวนลูกค้าที่ถูกปฏิเสธเก็บอยู่ใน object hotel ซึ่งมีเฉพาะตอนรันโปรแกรมจำลอง
# ถ้าไฟล์นี้อ่านจาก CSV อย่างเดียว ให้รายงานมูลค่าที่เสียจากการยกเลิกแทน
try:
    print(f"4. มีลูกค้าถูกปฏิเสธเพราะห้องเต็ม {len(hotel.rejected_log):,} ราย")
    print("   -> คือรายได้ที่หลุดมือ ควรใช้เป็นข้อมูลตัดสินใจขยายห้อง")
except NameError:
    lost = bookings[bookings["status"] == "CANCELLED"]["total_price"].sum()
    print(f"4. ใบจองที่ถูกยกเลิกคิดเป็นมูลค่า {lost:,.0f} บาท")
    print("   -> ควรใช้นโยบายมัดจำกับการจองผ่าน OTA ที่ยกเลิกบ่อยที่สุด")
print("=" * 62)

## ส่วนที่ 10 — สรุปผลเชิงธุรกิจ

In [ ]:
best = by_room_type.iloc[0]
best_night = by_room_type.sort_values("net_per_night", ascending=False).iloc[0]
ota = by_channel[by_channel["channel"] == "OTA"].iloc[0]
high = by_season[by_season["season"] == "High"].iloc[0]
low = by_season[by_season["season"] == "Low"].iloc[0]

print(f"1) ห้องที่ทำรายได้สุทธิรวมสูงสุด : {best['room_type']} "
      f"({best['net_revenue']:,.0f} บาท จาก {best['total_bookings']} ใบจอง)")
print(f"2) ห้องที่ให้รายได้ต่อคืนสูงสุด  : {best_night['room_type']} "
      f"({best_night['net_per_night']:,.0f} บาท/คืน)")
print(f"3) ช่องทาง OTA ยอดขาย {ota['gross_revenue']:,.0f} บาท "
      f"ค่าคอมมิชชัน {ota['commission_paid']:,.0f} บาท ({ota['commission_pct']:.1f}%)")
print(f"4) ราคาเฉลี่ยต่อคืน High season {high['avg_rate']:,.0f} บาท "
      f"เทียบ Low season {low['avg_rate']:,.0f} บาท")
print(f"5) ถูกปฏิเสธเพราะห้องเต็ม {len(hotel.rejected_log):,} ราย")

### สรุปผล 5 ข้อ

1. **ห้อง Deluxe ทำรายได้สุทธิรวมสูงที่สุด** เนื่องจากเป็นจุดที่ราคาต่อคืนอยู่ในระดับสูง
   พอสมควรและยังมีปริมาณการจองมาก ต่างจากห้อง Suite ที่ราคาสูงกว่าแต่มีเพียง 2 ห้อง
   จึงมีเพดานปริมาณจำกัด

2. **เมื่อพิจารณารายได้สุทธิต่อคืน ห้อง Suite ให้ผลตอบแทนสูงที่สุด** หากโรงแรมจะลงทุน
   เพิ่มห้อง ควรพิจารณาห้อง Suite และ Family มากกว่าห้อง Standard ตัวเลขรวมและ
   ตัวเลขต่อหน่วยจึงให้ข้อสรุปที่ต่างกันและต้องดูควบคู่กัน

3. **ช่องทาง OTA นำลูกค้าเข้ามามากที่สุดแต่มีต้นทุนสูงที่สุด** ทั้งค่าคอมมิชชัน 15%
   และอัตราการยกเลิกที่สูงเป็นสองเท่าของการจองตรง ข้อเสนอเชิงกลยุทธ์คือส่งเสริมให้
   ลูกค้าเดิมกลับมาจองผ่านเว็บไซต์ของโรงแรม ซึ่งมีค่าธรรมเนียมเพียง 2%

4. **ฤดูกาลมีผลต่อราคาอย่างชัดเจนแต่ไม่ทำให้ปริมาณการจองลดลงมาก** ราคาเฉลี่ยต่อคืน
   ในช่วง High season สูงกว่า Low season ประมาณ 40% ขณะที่จำนวนใบจองใกล้เคียงกัน
   แสดงว่ายังมีโอกาสปรับราคาขึ้นได้อีกในช่วงฤดูท่องเที่ยว

5. **ตัวชี้วัดของข้อมูลจำลองสอดคล้องกับตลาดจริง** ทั้งอัตราการเข้าพัก ADR RevPAR
   ระยะเวลาพักเฉลี่ย และอัตราการยกเลิกแยกตามช่องทาง อยู่ในช่วงเดียวกับข้อมูลตลาด
   โรงแรมไทยปี 2025 ทำให้ข้อเสนอทั้ง 4 ข้อข้างต้นสามารถนำไปใช้อ้างอิงได้จริง